# Transform Customer
transfoms the customer data and saves the cleaned result al a Silver table.

In [0]:

from pyspark.sql import functions as F

In [0]:
df = (
    spark.read
    .table("ecommerce_catalog.bronze.customers_raw")
    .withColumn(
        "is_imputed", 
        F.lit(None).cast("string")
        )
    .withColumn(
        "registration_date",
        F.coalesce(
            F.try_to_date("registration_date", "yyyy-MM-dd"),
            F.try_to_date("registration_date", "MM/dd/yyyy"),
            F.try_to_date("registration_date", "dd.MM.yyyy"),
            )
        )
    )
display(df)

In [0]:
df = df.dropDuplicates()

duplicate_values =( 
                   df.groupBy("customer_id")
                    .count()
                    .filter(F.col("count") > 1)
                    )


display(duplicate_values)

In [0]:
df = df.withColumn("customer_status", F.initcap(F.col("customer_status")))

display(df)

In [0]:
df_no_domain = df.filter(F.col("email").endswith("@"))

display(df_no_domain)

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists("ecommerce_catalog.silver.silver_customers"):
   dlt_obj = DeltaTable.forName(spark, "ecommerce_catalog.silver.silver_customers")

   (
       dlt_obj.alias("trg").merge(df.alias("src"),"trg.customer_id == src.customer_id")
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
        )

else:
    (
        df.write.format("delta")
        .mode("append")
        .saveAsTable("ecommerce_catalog.silver.silver_customers")
        )

### **Summary of Silver Transformation**
The customer data was cleaned and standardized for the Silver layer.

The following transformations were implemented:
- Complete duplicates were removed.
- `registration_date` was converted to a unified date format.
- Inconsistent casing in `customer_status` was standardized.
- The identified invalid email addresses were investigated. Since the missing domains cannot be unambiguously reconstructed from the available data, these values were intentionally left unmodified to avoid introducing potentially inaccurate information.

The cleaned and standardized customer data was subsequently persisted as a Silver table.